# TP4 : garanties de livraison et panne de broker

| Repère | Valeur |
|---|---|
| Moment | Jour 1, 16:00-17:00, après la démo du TP (15:55-16:00) ; puis démonstration D4 par le formateur (17:00-17:10) |
| Compétences | C7 Paramètres de fiabilité ; C8 Panne de broker et réplication |
| Slides de rappel | S36 (ISR, LEO, HW), S37 (`min.insync.replicas` × `acks`), S38 (élection et ELR), S40 (panne contre perte de quorum) |
| Version Markdown | `enonce.md`, mêmes étapes en ligne de commande seule |

Mode conseillé : `extended`, pour voir les alertes à l'étape 6.7. En mode `core`, cette étape est sautée.

**Mode d'emploi du notebook**

- Exécutez les cellules dans l'ordre avec `Maj+Entrée`. Après un redémarrage du noyau, les variables sont perdues : relancez la cellule d'outils, puis les cellules qui définissent les fonctions et les variables dont vous avez besoin. Ne relancez pas une cellule qui modifie le cluster (création d'un topic, arrêt d'un nœud) : elle échouerait, ou referait l'action.
- Chaque étape suit le même schéma : une explication, un **exemple en ligne de commande**, une **zone `# TODO`**, puis une **vérification**. Les commandes qui modifient le cluster sont affichées sans être exécutées (« ne pas lancer en plus du TODO »).
- Ce TP arrête un nœud : les exemples en ligne de commande redirigent les erreurs (`2>/dev/null`), car `$BS` contient encore l'adresse du nœud arrêté et chaque commande afficherait `Couldn't resolve server kafka-3:19092`.
- Pendant la panne, des lignes commençant par `%3|` ou `%4|` peuvent apparaître sous une cellule : c'est le journal de librdkafka, la bibliothèque du client Python, qui signale ses tentatives de connexion au nœud arrêté. Ce ne sont pas des erreurs du notebook.
- Les fonctions `end_offsets()` et `read()` du TP2 sont fournies dans la cellule suivante.

In [ ]:
# Connexion au lab : adresses des brokers, configuration du client Python, dossier de travail
import os
import time

from confluent_kafka.admin import AdminClient

BS = os.environ["BS"]  # listener INTERNAL, comme $BS dans le conteneur tools
PROMETHEUS = "http://prometheus:9090"
CONF = {"bootstrap.servers": BS, "log_level": 2}

# Les commandes docker compose et les fichiers config/ se lisent depuis le dossier lab/
LAB_DIR = os.environ["FORMATION_DIR"] + "/lab"
os.chdir(LAB_DIR)
os.environ["COMPOSE_PROGRESS"] = "plain"  # docker compose sans animation dans le notebook

In [ ]:
# Deux fonctions d'aide, écrites ici : elles ne font pas partie de confluent-kafka
def wait_for(condition, timeout=60, step=2, message="condition"):
    """Rappelle condition() jusqu'à obtenir une valeur vraie, et la renvoie."""
    start = time.time()
    while True:
        try:
            value = condition()
        except Exception:
            value = None
        if value:
            return value
        if time.time() - start >= timeout:
            raise TimeoutError("%s : non atteinte en %d s" % (message, timeout))
        time.sleep(step)

def table(lines, headers):
    """Affiche des lignes en tableau Markdown dans le notebook."""
    from IPython.display import Markdown, display

    md = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    md += ["| " + " | ".join(str(v).replace("|", "\\|") for v in line) + " |" for line in lines]
    display(Markdown("\n".join(md)))

In [ ]:
import threading
import time

import requests
from confluent_kafka import Consumer, ElectionType, KafkaException, Producer, TopicPartition
from confluent_kafka.admin import AlterConfigOpType, ConfigEntry, ConfigResource, ConfigSource, NewTopic, OffsetSpec


def end_offsets(topic):
    admin = AdminClient(CONF)
    num_partitions = len(admin.list_topics(topic, timeout=10).topics[topic].partitions)
    futures = admin.list_offsets({TopicPartition(topic, p): OffsetSpec.latest() for p in range(num_partitions)})
    return {tp.partition: f.result().offset for tp, f in sorted(futures.items(), key=lambda x: x[0].partition)}


def read(topic, partition, start, count):
    consumer = Consumer({"bootstrap.servers": BS, "group.id": "tp4-lecture",
                         "enable.auto.commit": False, "log_level": 4})
    consumer.assign([TopicPartition(topic, partition, start)])
    messages = []
    while len(messages) < count:
        batch = consumer.consume(count - len(messages), timeout=10)
        if not batch:
            break
        messages += [m for m in batch if m.error() is None]
    consumer.close()
    return messages


print("Bootstrap INTERNAL :", BS)

## 1. Objectifs

À la fin du TP, vous savez :

- fixer le nombre minimal de répliques synchronisées au bon niveau (cluster, topic) ;
- prédire le résultat d'une production selon `acks` et `min.insync.replicas` quand un broker manque ;
- diagnostiquer une panne de broker : partitions sous-répliquées, ISR, ELR, quorum, brokers enregistrés ;
- vérifier qu'un producteur `acks=all` n'a perdu aucun message pendant la panne ;
- ramener le cluster à l'état nominal : rattrapage des répliques et élection des leaders préférés.

## 2. Prérequis

- TP1 à TP3 terminés, cluster sain : `scripts/validate.sh` au vert.
- Notions : slides S35 à S41.

## 3. Architecture utilisée

```
  Avant la panne                          Pendant la panne (kafka-3 arrêté)

  kafka-1   kafka-2   kafka-3             kafka-1   kafka-2   kafka-3
   [L]       [F]       [F]   payments-1    [L]       [F]        X
   [F]       [L]       [F]   payments-2    [F]       [L]        X
   [F]       [F]       [L]   payments-0    [L]       [F]        X   <- nouveau leader élu

  ISR = {1,2,3}                           ISR = {1,2}  (2 >= min.insync.replicas = 2 : écriture acceptée)
  refunds : min.insync.replicas = 3       ISR = {1,2}  (2 <  3 : écriture acks=all refusée)

  Quorum KRaft : 3 votants. Avec 2 votants sur 3, la majorité est conservée.
```

| Topic | Partitions | RF | `min.insync.replicas` |
|---|---|---|---|
| `payments` | 3 | 3 | 2 (hérité du cluster) |
| `refunds` | 1 | 3 | 3 (posé sur le topic) |

## 4. Contexte métier

Pour NovaShop, un paiement perdu est un incident grave : le client est débité et la commande n'est jamais expédiée. La direction exige que le flux `payments` résiste à la perte d'un nœud, sans perte et sans interruption. Pour les remboursements, l'équipe Finance a demandé une confirmation par les 3 répliques.

Vous devez configurer ces exigences, puis les prouver en arrêtant un nœud en pleine charge. Le bilan de l'exercice s'affiche en fin de notebook.

## 5. Énoncé

1. Exigez au moins 2 répliques synchronisées par défaut sur tout le cluster.
2. Créez `payments` et `refunds` selon le tableau de la section 4.
3. Arrêtez `kafka-3` pendant qu'un producteur `acks=all` écrit 100 paiements par seconde.
4. Diagnostiquez l'état du cluster, prédisez puis vérifiez le comportement des productions selon `acks`.
5. Redémarrez `kafka-3`, ramenez le cluster à l'état nominal et prouvez qu'aucun paiement n'a été perdu.

## 6. Étapes

### 6.1 Fixer le minimum ISR du cluster

`min.insync.replicas` fixe le nombre minimal de répliques synchronisées pour qu'une écriture `acks=all` soit acceptée. Depuis Kafka 4.1, l'ELR (Eligible Leader Replicas) est actif et impose une valeur unique pour tout le cluster, modifiable aussi par topic.

**Exemple en ligne de commande** : la configuration dynamique par défaut des brokers.

In [ ]:
!docker exec kafka-tools kafka-configs.sh --bootstrap-server $BS --describe --entity-type brokers --entity-default

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-configs.sh --bootstrap-server $BS --alter --entity-type brokers --entity-name 1 \
  --add-config min.insync.replicas=2
kafka-configs.sh --bootstrap-server $BS --alter --entity-type brokers --entity-default \
  --add-config min.insync.replicas=2
```

In [ ]:
# TODO 6.1 : fixer min.insync.replicas=2
# 1. En Python, demandez min.insync.replicas=2 sur le seul broker 1 :
#      ConfigResource("broker", "1", incremental_configs=[ConfigEntry("min.insync.replicas", "2",
#                     incremental_operation=AlterConfigOpType.SET)])
#    avec admin.incremental_alter_configs([...]) ; rangez le message d'erreur (e.args[0].str()) dans broker_refusal.
# 2. Le client Python ne sait pas viser la configuration par défaut de tous les brokers
#    (ConfigResource("broker", "") lève ValueError). Faites ce réglage avec l'outil Java, par une cellule `!docker exec kafka-tools ...` :
#    l'option --entity-default remplace --entity-name.
broker_refusal = None

print("refus_broker :", broker_refusal)

In [ ]:
# Vérification 6.1
assert broker_refusal is not None and "ELR" in broker_refusal, "À revoir : réglage par broker refusé : %s" % broker_refusal
print("OK  réglage par broker refusé : %s" % broker_refusal)
default = !docker exec kafka-tools kafka-configs.sh --bootstrap-server $BS --describe --entity-type brokers --entity-default 2>&1
default = default.n
assert "min.insync.replicas=2 " in default, "À revoir : min.insync.replicas=2 pour tout le cluster"
print("OK  min.insync.replicas=2 pour tout le cluster")

**Résultat attendu** :

- valeur de départ : `min.insync.replicas=1`, source `DYNAMIC_DEFAULT_BROKER_CONFIG` ;
- le réglage sur le seul broker 1 est refusé : `Broker-level min.insync.replicas cannot be altered while ELR is enabled.` ;
- le réglage par défaut du cluster réussit : `Completed updating default config for brokers in the cluster.`

### 6.2 Créer `payments` et `refunds`

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
kafka-topics.sh --bootstrap-server $BS --create --topic payments --partitions 3 --replication-factor 3
kafka-topics.sh --bootstrap-server $BS --create --topic refunds --partitions 1 --replication-factor 3 \
  --config min.insync.replicas=3
```

In [ ]:
# TODO 6.2 : créer payments et refunds, puis lire l'origine de leur min.insync.replicas
# 1. Créez payments (3 partitions, RF 3) et refunds (1 partition, RF 3, min.insync.replicas=3 :
#    NewTopic(..., config={...})).
# 2. Pour chacun des deux topics, lisez l'entrée min.insync.replicas de describe_configs (voir TP2)
#    et rangez dans min_isr {topic: (valeur, nom de l'origine)} ; nom de l'origine : ConfigSource(entry.source).name.
min_isr = {}

print(min_isr)

In [ ]:
# Vérification 6.2
assert min_isr.get("payments") == ("2", "DYNAMIC_DEFAULT_BROKER_CONFIG"), "À revoir : payments hérité 2 du cluster"
print("OK  payments hérité 2 du cluster")
assert min_isr.get("refunds") == ("3", "DYNAMIC_TOPIC_CONFIG"), "À revoir : refunds exige 3 répliques"
print("OK  refunds exige 3 répliques")

In [ ]:
!docker exec kafka-tools bash -c "kafka-topics.sh --bootstrap-server $BS --describe --topic 'payments|refunds'"

**Résultat attendu** : `payments` hérite `min.insync.replicas=2` du cluster (`DYNAMIC_DEFAULT_BROKER_CONFIG`) ; `refunds` a sa propre valeur, 3. Chaque partition a 3 répliques, toutes dans l'ISR.

### 6.3 Produire en `acks=all`

Avec `acks=all`, le leader ne confirme un message qu'une fois écrit par toutes les répliques de l'ISR, et seulement si l'ISR compte au moins `min.insync.replicas` répliques.

Équivalent en ligne de commande (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
seq 1 100 | sed 's/^/PAY-/' | kafka-console-producer.sh --bootstrap-server $BS --topic payments \
  --request-required-acks all
```

In [ ]:
# TODO 6.3 : envoyer PAY-1 à PAY-100 dans payments, sans clé, en acks=all
# Créez un Producer avec "acks": "all", envoyez les 100 messages, puis flush().
# Comptez les confirmations sans erreur dans confirmed_73 (rappel on_delivery, voir TP2).
admin = AdminClient(CONF)
confirmed_73 = 0

print(confirmed_73, "confirmés ;", end_offsets("payments") if "payments" in admin.list_topics(timeout=10).topics else "")

In [ ]:
# Vérification 6.3
assert confirmed_73 == 100, "À revoir : 100 paiements confirmés"
print("OK  100 paiements confirmés")
assert sum(end_offsets("payments").values()) == 100, "À revoir : 100 messages dans payments"
print("OK  100 messages dans payments")

**Résultat attendu** : 100 confirmations. Sans clé, les messages se concentrent dans une partition (partitionneur « sticky », TP2).

### 6.4 Charge continue et arrêt de `kafka-3`

La classe `Load` ci-dessous est un producteur qui tourne en arrière-plan : il envoie `count` messages à `rate` messages par seconde et compte les confirmations (`confirmed`) et les erreurs définitives (`errors`). Elle remplace `kafka-producer-perf-test.sh` de la version Markdown.

In [ ]:
class Load:
    """Producteur de charge en arrière-plan : count messages, à rate messages par seconde."""

    def __init__(self, topic, count, rate, config):
        self.confirmed = 0
        self.errors = []
        self.producer = Producer({"bootstrap.servers": BS, "log_level": 4, **config})
        self._thread = threading.Thread(target=self._send, args=(topic, count, rate), daemon=True)
        self._thread.start()

    def _report(self, err, msg):
        if err is None:
            self.confirmed += 1
        else:
            self.errors.append(err)

    def _send(self, topic, count, rate):
        start = time.time()
        for i in range(count):
            # Cadence régulière : le message i part à start + i / rate
            pause = start + i / rate - time.time()
            if pause > 0:
                time.sleep(pause)
            self.producer.produce(topic, value="CHARGE-%d" % (i + 1), on_delivery=self._report)
            self.producer.poll(0)
        self.producer.flush(120)

    def running(self):
        return self._thread.is_alive()

    def __repr__(self):
        return "Charge(%d confirmés, %d erreurs%s)" % (self.confirmed, len(self.errors),
                                                      ", en cours" if self.running() else ", terminée")

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
docker compose exec -d tools bash -c 'kafka-producer-perf-test.sh --bootstrap-server $BS --topic payments \
  --num-records 6000 --throughput 100 --record-size 200 --command-property acks=all \
  > /tmp/perf-payments.log 2>&1'
sleep 5
time docker compose stop kafka-3
```

**Exécutez la cellule `TODO` une seule fois**, d'un seul bloc : le délai de 5 s entre la charge et l'arrêt compte.

In [ ]:
# TODO 6.4 : lancer 6000 paiements à 100 par seconde, puis arrêter kafka-3 5 s plus tard
#   1. load = Load("payments", 6000, 100, {"acks": "all", "enable.idempotence": True})
#      L'idempotence, active par défaut chez les producteurs Java depuis Kafka 3.0, empêche les doublons
#      quand un envoi est rejoué après un changement de leader.
#   2. Attendez 5 s (time.sleep).
#   3. Arrêtez kafka-3 avec `!docker compose stop kafka-3` ; rangez la durée de l'arrêt dans downtime.
load = None
downtime = None

print(load, "- arrêt en", downtime, "s")

In [ ]:
# Vérification 6.4
assert load is not None and load.running(), "À revoir : charge en cours : %s" % load
print("OK  charge en cours : %s" % load)
output = !docker compose ps --format '{{{{.Name}}}}' kafka-3 2>&1
assert output.n == "", "À revoir : kafka-3 arrêté"
print("OK  kafka-3 arrêté")

**Résultat attendu** : `docker compose stop kafka-3` dure environ 3 s. La charge continue, sans erreur.

### 6.5 Diagnostiquer la panne

**Exemples en ligne de commande** : partitions sous-répliquées, `payments`, partitions au minimum ISR et sous le minimum, quorum, brokers enregistrés.

In [ ]:
!docker exec kafka-tools bash -c "kafka-topics.sh --bootstrap-server $BS --describe --under-replicated-partitions 2>/dev/null | grep -c Partition"
!docker exec kafka-tools bash -c "kafka-topics.sh --bootstrap-server $BS --describe --topic payments 2>/dev/null"
!docker exec kafka-tools bash -c "kafka-topics.sh --bootstrap-server $BS --describe --at-min-isr-partitions 2>/dev/null | grep -c Partition"
!docker exec kafka-tools bash -c "kafka-topics.sh --bootstrap-server $BS --describe --under-min-isr-partitions 2>/dev/null"
!docker exec kafka-tools bash -c "kafka-metadata-quorum.sh --bootstrap-server $BS describe --replication 2>/dev/null"
!docker exec kafka-tools bash -c "kafka-cluster.sh list-endpoints --bootstrap-server $BS 2>/dev/null"

In [ ]:
# TODO 6.5 : le même diagnostic en Python
# 1. Écrivez partition_state() : liste de tuples (topic, partition, leader, replicas, isr) pour toutes
#    les partitions du cluster. admin.list_topics(timeout=10).topics est un dictionnaire {name: topic} ;
#    topic.partitions est un dictionnaire {numero: partition}, et chaque partition a les attributs
#    id, leader, replicas et isrs (listes d'identifiants).
# 2. Rangez dans urp le nombre de partitions sous-répliquées (isr plus courte que répliques).
# 3. Rangez dans payments_state les lignes de partition_state() qui concernent payments, triées.
# 4. Rangez dans live_brokers la liste triée des identifiants de admin.describe_cluster().result().nodes.
def partition_state():
    return []


urp = None
payments_state = []
live_brokers = []

print("URP :", urp, "- brokers :", live_brokers)
table(payments_state, ["topic", "partition", "leader", "répliques", "isr"])

In [ ]:
# Vérification 6.5
assert urp is not None and urp >= 50, "À revoir : partitions sous-répliquées : %s (dont les 50 de __consumer_offsets)" % urp
print("OK  partitions sous-répliquées : %s (dont les 50 de __consumer_offsets)" % urp)
assert len(payments_state) == 3 and all(l[2] != 3 and 3 not in l[4] for l in payments_state), "À revoir : payments : aucun leader ni aucune réplique synchronisée sur kafka-3"
print("OK  payments : aucun leader ni aucune réplique synchronisée sur kafka-3")
assert live_brokers == [1, 2], "À revoir : brokers enregistrés : %s" % live_brokers
print("OK  brokers enregistrés : %s" % live_brokers)

**Résultat attendu** :

| Mesure | Résultat |
|---|---|
| Partitions sous-répliquées | plusieurs dizaines : toutes celles qui avaient une réplique sur `kafka-3` |
| `payments` | aucun leader sur 3 ; ISR `[1, 2]` ; liste des répliques inchangée |
| Au minimum ISR | presque autant que les sous-répliquées |
| Sous le minimum ISR | `refunds`, avec `Isr: 2,1` (ou `1,2`) et `Elr: 3` |
| Quorum | `kafka-3` toujours listé comme `Follower`, avec un `Lag` qui augmente ; un leader existe |
| Brokers enregistrés | 1 et 2 seulement |

### 6.6 Prédire, puis produire

Remplissez d'abord le dictionnaire `predictions` ci-dessous (« acceptée » ou « refusée »), **puis** seulement la cellule `TODO`.

In [ ]:
predictions = {
    "payments, 10 messages, acks=all": "",
    "refunds, REF-1, acks=all": "",
    "refunds, REF-2, acks=1": "",
}

Équivalent en ligne de commande (**ne pas lancer en plus du TODO**) :

```bash
# [tools]
seq 101 110 | sed 's/^/PAY-/' | kafka-console-producer.sh --bootstrap-server $BS --topic payments \
  --request-required-acks all
echo REF-1 | kafka-console-producer.sh --bootstrap-server $BS --topic refunds --request-required-acks all \
  --command-property delivery.timeout.ms=15000 --command-property request.timeout.ms=5000
echo REF-2 | kafka-console-producer.sh --bootstrap-server $BS --topic refunds --request-required-acks 1
```

In [ ]:
# TODO 6.6 : les trois productions
# Écrivez produce(topic, values, acks) : elle envoie les valeurs avec un Producer configuré par
#   {"bootstrap.servers": BS, "acks": acks, "retries": 3, "message.timeout.ms": 15000, "log_level": 4}
# et renvoie la liste des résultats, un par message : None si le message est écrit, sinon err.name().
# retries=3 : trois nouvelles tentatives, comme le producteur console de Java. Sans cette limite,
# librdkafka réessaie jusqu'à message.timeout.ms et l'erreur finale masque la cause.
def produce(topic, values, acks):
    return []


results = {
    "payments, 10 messages, acks=all": produce("payments", ["PAY-%d" % i for i in range(101, 111)], "all"),
    "refunds, REF-1, acks=all": produce("refunds", ["REF-1"], "all"),
    "refunds, REF-2, acks=1": produce("refunds", ["REF-2"], 1),
}

table([(name, predictions[name], results[name]) for name in predictions], ["Production", "Prédiction", "Résultat"])

In [ ]:
# Vérification 6.6
assert results["payments, 10 messages, acks=all"] == [None] * 10, "À revoir : payments en acks=all : accepté"
print("OK  payments en acks=all : accepté")
assert results["refunds, REF-1, acks=all"] == ["NOT_ENOUGH_REPLICAS"], "À revoir : refunds en acks=all : refusé"
print("OK  refunds en acks=all : refusé")
assert results["refunds, REF-2, acks=1"] == [None], "À revoir : refunds en acks=1 : accepté"
print("OK  refunds en acks=1 : accepté")

**Résultat attendu** :

| Production | Résultat | Raison |
|---|---|---|
| `payments`, `acks=all` | Acceptée | ISR = 2 ≥ minimum 2 |
| `refunds`, `acks=all` | Refusée, `NOT_ENOUGH_REPLICAS` | ISR = 2 < minimum 3 |
| `refunds`, `acks=1` | Acceptée | Le minimum ISR ne s'applique qu'à `acks=all` |

### 6.7 Alertes

Prometheus évalue les règles d'alerte du lab toutes les 15 s. Une alerte passe par l'état `pending` pendant son délai `for`, puis `firing`. Interface : http://localhost:9090, menu « Alerts ».

**Exemple en ligne de commande** : l'API de Prometheus, depuis le conteneur `tools` (la version Markdown utilise `curl` sur l'hôte, avec `localhost:9090`). Juste après l'arrêt, la liste peut encore être vide : les alertes passent d'abord par `pending`.

In [ ]:
!docker exec kafka-tools bash -c 'wget -qO- http://prometheus:9090/api/v1/alerts | grep -o '"'"'"alertname":"[A-Za-z]*"'"'"' | sort | uniq -c || echo '"'"'Aucune alerte pour le moment, ou Prometheus absent (mode core)'"'"''

In [ ]:
# TODO 6.7 : relever les alertes actives
# 1. Écrivez active_alerts() : ensemble des noms des alertes à l'état "firing".
#    requests.get(PROMETHEUS + "/api/v1/alerts", timeout=5).json()["data"]["alerts"] est une liste de
#    dictionnaires ; chacun a "state" et "labels" (dictionnaire, dont "alertname").
# 2. Attendez (wait_for, timeout=150, step=5) que KafkaNodeDown soit active, puis rangez
#    active_alerts() dans alerts.
# En mode core (sans Prometheus), passez à l'étape suivante.
def active_alerts():
    return set()


alerts = set()

print(sorted(alerts))

In [ ]:
# Vérification 6.7
try:
    requests.get(PROMETHEUS + "/-/ready", timeout=3)
    assert "KafkaNodeDown" in alerts, "À revoir : alerte KafkaNodeDown active"
    print("OK  alerte KafkaNodeDown active")
    assert {"KafkaUnderReplicatedPartitions", "KafkaUnderMinIsrPartitions"} <= alerts, "À revoir : alertes de réplication actives : %s" % sorted(alerts)
    print("OK  alertes de réplication actives : %s" % sorted(alerts))
except requests.RequestException:
    print("Prometheus absent (mode core) : étape 6.7 sautée")

**Résultat attendu** : environ une minute après l'arrêt, 4 alertes `firing` : `KafkaNodeDown`, `KafkaUnderMinIsrPartitions`, `KafkaUnderReplicatedPartitions` et `KafkaFencedBrokers`.

### 6.8 Redémarrer `kafka-3` et revenir à l'état nominal

Au redémarrage, les répliques de `kafka-3` rattrapent leur retard puis rentrent dans l'ISR. Mais `kafka-3` ne redevient leader de rien : il faut une **élection des leaders préférés** (le premier broker de chaque liste de répliques).

Équivalent en ligne de commande du `TODO` (**ne pas lancer en plus du TODO**) :

```bash
# [hôte]
docker compose start kafka-3
# [tools]
kafka-leader-election.sh --bootstrap-server $BS --election-type preferred --all-topic-partitions
```

In [ ]:
# TODO 6.8 : redémarrer kafka-3, mesurer le retour de l'ISR, puis élire les leaders préférés
#   1. Notez début, redémarrez kafka-3 (`!docker compose start kafka-3`), puis attendez
#      (wait_for, timeout=120) qu'il n'y ait plus de partition sous-répliquée. Rangez la durée dans urp_duration.
#   2. Rangez dans before_election les tuples (partition, leader, leader préféré) de payments ;
#      le leader préféré est le premier élément de la liste des répliques.
#   3. Lancez l'élection : admin.elect_leaders(ElectionType.PREFERRED, None).result()
#      (None : toutes les partitions). Le résultat est un dictionnaire {TopicPartition: erreur}.
#      Rangez son nombre d'éléments dans elected_count.
#   4. Rangez dans after_election les mêmes tuples qu'au point 2.
urp_duration = None
before_election = after_election = []
elected_count = None

print("URP à 0 en", urp_duration, "s ;", elected_count, "partitions élues")
table(before_election, ["partition", "leader", "préféré"])
table(after_election, ["partition", "leader", "préféré"])

In [ ]:
# Vérification 6.8
assert partition_state() and all(len(l[4]) == len(l[3]) for l in partition_state()), "À revoir : plus aucune partition sous-répliquée"
print("OK  plus aucune partition sous-répliquée")
assert any(leader != preferred for _, leader, preferred in before_election), "À revoir : avant l'élection, un leader de payments n'est pas le préféré"
print("OK  avant l'élection, un leader de payments n'est pas le préféré")
assert after_election and all(leader == preferred for _, leader, preferred in after_election), "À revoir : après l'élection, chaque partition de payments a son leader préféré"
print("OK  après l'élection, chaque partition de payments a son leader préféré")

In [ ]:
!docker exec kafka-tools bash -c "kafka-topics.sh --bootstrap-server $BS --describe --topic 'payments|refunds'"

**Résultat attendu** :

- plus aucune partition sous-répliquée quelques secondes après le démarrage ;
- avant l'élection, la partition de `payments` dont le leader préféré est 3 garde son leader de secours ;
- l'élection concerne plusieurs dizaines de partitions, puis chaque partition de `payments` retrouve son leader préféré.

### 6.9 Défi : prouver qu'aucun paiement n'est perdu

Combien de messages `payments` devrait-il contenir ? Prouvez-le avec deux sources indépendantes : les offsets du topic et le bilan du producteur de charge. Vérifiez aussi le contenu de `refunds`.

In [ ]:
# TODO 6.9 (défi) : le bilan
#   1. Attendez la fin de la charge : wait_for(lambda: not load.running(), timeout=120).
#   2. expected : nombre de messages que payments doit contenir (envois confirmés des étapes 6.3, 6.4 et 6.6).
#   3. total_payments : somme des offsets de fin de payments.
#   4. refunds_content : liste des valeurs (décodées) de la partition 0 de refunds, avec read().
expected = total_payments = None
refunds_content = []

print("attendu :", expected, "- dans le topic :", total_payments, "- producteur de charge :", load)
print("refunds :", refunds_content)

In [ ]:
# Vérification 6.9
assert load is not None and load.confirmed == 6000 and not load.errors, "À revoir : 6000 envois de charge confirmés, aucune erreur définitive"
print("OK  6000 envois de charge confirmés, aucune erreur définitive")
assert expected == 6110 and total_payments == expected, "À revoir : " + ("payments contient %s messages pour %s attendus"
         % (total_payments, expected))
print("OK  " + ("payments contient %s messages pour %s attendus"
         % (total_payments, expected)))
assert refunds_content == ["REF-2"], "À revoir : refunds ne contient que REF-2"
print("OK  refunds ne contient que REF-2")

**Résultat attendu** : `payments` contient 100 + 6000 + 10 = 6110 messages, autant que les confirmations reçues. Le producteur de charge n'a aucune erreur définitive. `refunds` contient seulement `REF-2`.

## Bilan de l'exercice

In [ ]:
output = !docker exec kafka-tools kafka-configs.sh --bootstrap-server $BS --describe --entity-type brokers --entity-default 2>&1
table([
    ("min.insync.replicas du cluster", "2" if "min.insync.replicas=2 " in output.n else "à revoir", "6.1"),
    ("Arrêt de kafka-3", "%s s" % downtime if downtime else "à compléter", "6.4"),
    ("Partitions sous-répliquées pendant la panne", urp if urp is not None else "à compléter", "6.5"),
    ("Productions (payments all, refunds all, refunds 1)",
     " / ".join("OK" if r and all(x is None for x in r) else str(r) for r in results.values()), "6.6"),
    ("Alertes actives", ", ".join(sorted(alerts)) or "aucune relevée", "6.7"),
    ("Retour de l'ISR après redémarrage", "%s s" % urp_duration if urp_duration else "à compléter", "6.8"),
    ("Messages dans payments / attendus", "%s / %s" % (total_payments, expected), "6.9"),
], ["Contrôle", "Valeur", "Étape"])

## 7. Questions de réflexion

1. Pourquoi le minimum ISR ne peut-il pas être fixé broker par broker quand l'ELR est actif ?
2. Avec RF 3 et `min.insync.replicas=2`, combien de brokers peut-on perdre sans bloquer les producteurs `acks=all` ? Et sans perdre de données ?
3. `refunds` avec `min.insync.replicas=3` : quel est le prix de cette exigence ? Que conseilleriez-vous à l'équipe Finance ?
4. Le message `REF-2` écrit en `acks=1` pendant la panne est-il en sécurité ? Pourquoi ?
5. Que représente `Elr: 3` sur `refunds` ?
6. Pourquoi faut-il une élection des leaders préférés après le retour d'un broker ? Que se passe-t-il si on ne la lance pas ?
7. Quelle différence entre perdre un broker (ce TP) et perdre le quorum (démonstration D4) ?